# Evaluate TTA (OOF) for ConvNeXtSplitCropAuxRegressor
---
- split-crop入力 (B,2,C,H,W) 対応
- cropアンサンブル（NUM_CROPS）対応
- TTA（flip）対応（5D/6D対応）
- weighted_r2_oof は「全OOF結合して1回」で計算（参加者の言うCVに近い）
- 平均はデフォルト raw 空間（Kaggle推論と同じ）

In [1]:
# =========================================================
# 0. Setup
# =========================================================
import gc
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image, ImageFile

# 壊れ気味画像の読み込みを許容（それでもダメなら例外）
ImageFile.LOAD_TRUNCATED_IMAGES = True

In [2]:
# =========================================================
# 0. ユーザー設定（ここだけ触ればOK）
# =========================================================
PROJECT_DIR = Path("/mnt/nfs/home/hidebu/study/CSIRO---Image2Biomass-Prediction")
SRC_DIR = PROJECT_DIR / "src"

# 評価したい exp を指定（experiments/<EXP>）
EXP = "103_train_exp037"
EXP_DIR = PROJECT_DIR / "experiments" / EXP
CFG_PATH = EXP_DIR / "yaml" / "config.yaml"

# 評価するTTA
TTA_MODES = ["none", "hflip", "hflip_vflip"]

# crop設定（推論に寄せたい場合は center 推奨）
# - "center": 常に中央crop（推論再現性◎）
# - "grid"  : 中央 + 4隅（NUM_CROPS>1で有効）
# - "random": 画像ごとseed固定の疑似ランダム（再現性あり）
CROP_MODE = "center"
NUM_CROPS = 1  # 1なら通常。2～5でcropアンサンブル（計算コスト増）

# TTA平均空間
# - "raw":  各viewの log1p を expm1→raw に戻して raw 平均（推論と同じ）
# - "log":  log1pのまま平均 → 最後に expm1（比較用）
AVG_SPACE = "raw"

# DataLoader
BATCH_SIZE = 8          # split-cropは重いのでまず8推奨
NUM_WORKERS = 0         # OOFは再現性優先で0推奨
PIN_MEMORY = True

# AMP
USE_AMP = True

# log1p -> raw 変換時の安全クリップ
LOG_CLIP_MIN = -20.0
LOG_CLIP_MAX = 20.0

In [3]:
# =========================================================
# 1. import (project modules)
# =========================================================
assert SRC_DIR.exists(), SRC_DIR
import sys
sys.path.append(str(SRC_DIR))

from omegaconf import OmegaConf
from utils.metric import global_weighted_r2_score, r2_per_target
from models.convnext_splitcrop_aux_regressor import ConvNeXtSplitCropAuxRegressor

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

DEVICE: cuda


In [4]:
# =========================================================
# 2. Config / Data load
# =========================================================
assert CFG_PATH.exists(), CFG_PATH
cfg = OmegaConf.load(CFG_PATH)
print("[INFO] Loaded cfg:", CFG_PATH)

# 学習で使った pivot を読む（Fold列入り）
pp_dir = Path(str(cfg.pp_dir)) / str(cfg.preprocess_ver)
pivot_path = pp_dir / str(cfg.pivot_csv_name)
assert pivot_path.exists(), pivot_path

df = pd.read_csv(pivot_path)
print("[INFO] pivot df:", df.shape)

fold_col = str(cfg.fold_col)
target_cols = list(cfg.target_cols)

# folds は cfg に保存されている想定。無い場合は df から推定。
if hasattr(cfg, "folds") and cfg.folds is not None:
    folds = list(cfg.folds)
else:
    folds = sorted(df[fold_col].dropna().unique().tolist())

print("[INFO] folds:", folds)
print("[INFO] targets:", target_cols)

# metric weights
weights = np.asarray(list(cfg.metric.weights), dtype=np.float64)
print("[INFO] metric weights:", weights)


[INFO] Loaded cfg: /mnt/nfs/home/hidebu/study/CSIRO---Image2Biomass-Prediction/experiments/103_train_exp037/yaml/config.yaml
[INFO] pivot df: (357, 36)
[INFO] folds: [0, 1, 2]
[INFO] targets: ['Dry_Green_g', 'Dry_Clover_g', 'Dry_Dead_g', 'GDM_g', 'Dry_Total_g']
[INFO] metric weights: [0.1 0.1 0.1 0.2 0.5]


In [5]:
# =========================================================
# 3. Transform（split-crop validに合わせる）
# =========================================================
def build_splitcrop_valid_transform(cfg: Any) -> A.Compose:
    """split-crop用のvalid transform（Normalize + ToTensorV2）を作る。

    Args:
        cfg: OmegaConf。cfg.normalize.mean/std を参照する。

    Returns:
        Albumentations Compose。
    """
    mean = list(cfg.normalize.mean)
    std = list(cfg.normalize.std)
    return A.Compose([A.Normalize(mean=mean, std=std), ToTensorV2()])


VALID_TFM = build_splitcrop_valid_transform(cfg)

In [6]:
# =========================================================
# 4. Dataset（推論に寄せた deterministic split-crop）
# =========================================================
class OofSplitCropDataset(Dataset):
    """OOF評価用 split-crop Dataset（推論に寄せたcropを生成）。

    - 画像を左右に分割し、それぞれからcropして (2,C,H,W) を返す
    - NUM_CROPS>1 なら (N,2,C,H,W) を返す（推論側で平均する）
    - target は cfg.use_log1p_target に合わせて log1p or raw を返す

    Args:
        df: pivot df（image_id, image_path, target_cols を含む）
        image_root: 画像ルート（cfg.input_dir）
        target_cols: 目的変数列
        transform: Albumentations transform（Normalize + ToTensorV2）
        use_log1p_target: Trueなら target を log1p にして返す
        crop_size: cropサイズ（cfg.crop_size）
        crop_mode: "center" | "grid" | "random"
        num_crops: 1以上
        base_seed: randomモードのseed
    """

    def __init__(
        self,
        df: pd.DataFrame,
        image_root: Path,
        target_cols: List[str],
        transform: A.Compose,
        use_log1p_target: bool,
        crop_size: int,
        crop_mode: str,
        num_crops: int,
        base_seed: int = 1129,
    ) -> None:
        self.df = df.reset_index(drop=True)
        self.image_root = Path(image_root)
        self.target_cols = list(target_cols)
        self.transform = transform
        self.use_log1p_target = bool(use_log1p_target)

        self.crop_size = int(crop_size)
        self.crop_mode = str(crop_mode).lower()
        self.num_crops = int(num_crops)
        self.base_seed = int(base_seed)

        self.ids = self.df["image_id"].astype(str).values
        self.paths = self.df["image_path"].astype(str).values

        y = self.df[self.target_cols].values.astype(np.float32)
        if self.use_log1p_target:
            y = np.log1p(np.clip(y, 0.0, None))
        self.targets = y

        if self.num_crops < 1:
            raise ValueError("num_crops must be >= 1")

    @staticmethod
    def _load_rgb(path: Path) -> np.ndarray:
        """RGBで読み込み、HWC uint8 を返す。"""
        with Image.open(path) as img:
            img = img.convert("RGB")
            return np.asarray(img, dtype=np.uint8)

    @staticmethod
    def _split_lr(img: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """左右に分割して (left, right) を返す。"""
        h, w, _ = img.shape
        mid = w // 2
        return img[:, :mid, :], img[:, mid:, :]

    @staticmethod
    def _safe_crop(img: np.ndarray, top: int, left: int, crop_size: int) -> np.ndarray:
        """はみ出しを安全に処理して crop_size のHWCを返す（足りない場合はpad）。"""
        h, w, c = img.shape
        pad_h = max(crop_size - h, 0)
        pad_w = max(crop_size - w, 0)
        if pad_h > 0 or pad_w > 0:
            img = np.pad(img, ((0, pad_h), (0, pad_w), (0, 0)), mode="constant", constant_values=0)
            h, w, c = img.shape

        max_top = max(h - crop_size, 0)
        max_left = max(w - crop_size, 0)
        top = int(np.clip(top, 0, max_top))
        left = int(np.clip(left, 0, max_left))
        return img[top : top + crop_size, left : left + crop_size, :]

    def _topleft_list(self, h: int, w: int, idx: int) -> List[Tuple[int, int]]:
        """crop_mode に応じた (top,left) を num_crops 個返す。"""
        cs = self.crop_size
        max_top = max(h - cs, 0)
        max_left = max(w - cs, 0)

        if self.crop_mode == "center":
            t = max_top // 2
            l = max_left // 2
            return [(t, l)] * self.num_crops

        if self.crop_mode == "grid":
            points = [
                (0, 0),
                (0, max_left),
                (max_top, 0),
                (max_top, max_left),
                (max_top // 2, max_left // 2),
            ]
            out = []
            for i in range(self.num_crops):
                out.append(points[i % len(points)])
            return out

        if self.crop_mode == "random":
            rng = np.random.default_rng(self.base_seed + idx * 10007)
            out = []
            for _ in range(self.num_crops):
                t = int(rng.integers(0, max_top + 1)) if max_top > 0 else 0
                l = int(rng.integers(0, max_left + 1)) if max_left > 0 else 0
                out.append((t, l))
            return out

        raise ValueError(f"Unknown crop_mode: {self.crop_mode}")

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        image_id = self.ids[idx]
        img_path = self.image_root / self.paths[idx]

        if not img_path.exists():
            raise FileNotFoundError(f"image not found: {img_path}")

        img = self._load_rgb(img_path)
        left, right = self._split_lr(img)

        hl, wl, _ = left.shape
        hr, wr, _ = right.shape

        tl_list_l = self._topleft_list(hl, wl, idx)
        tl_list_r = self._topleft_list(hr, wr, idx)

        crops = []
        for (t0, l0), (t1, l1) in zip(tl_list_l, tl_list_r):
            lc = self._safe_crop(left, t0, l0, self.crop_size)
            rc = self._safe_crop(right, t1, l1, self.crop_size)

            t_l = self.transform(image=lc)["image"]   # (C,H,W)
            t_r = self.transform(image=rc)["image"]   # (C,H,W)
            crops.append(torch.stack([t_l, t_r], dim=0))  # (2,C,H,W)

        x = crops[0] if self.num_crops == 1 else torch.stack(crops, dim=0)  # (2,C,H,W) or (N,2,C,H,W)
        y = torch.from_numpy(self.targets[idx])  # (K,)
        return {"id": image_id, "image": x, "target": y}

In [7]:
# =========================================================
# 5. Model / ckpt loader
# =========================================================
def is_aux_on(aux_cfg: Any, head_name: str) -> bool:
    """aux head が有効か判定する（enabled & weight>0）。

    Args:
        aux_cfg: cfg.aux（OmegaConf）または None
        head_name: "species" | "ndvi" | "height"

    Returns:
        有効なら True。
    """
    if aux_cfg is None:
        return False
    if not bool(getattr(aux_cfg, "enabled", False)):
        return False
    head = getattr(aux_cfg, head_name, None)
    if head is None:
        return False
    return bool(getattr(head, "enabled", False)) and float(getattr(head, "weight", 0.0)) > 0.0


def compute_num_species(df: pd.DataFrame, aux_cfg: Any) -> int:
    """species head用のクラス数をpivot dfから作る（trainingと同じ思想）。"""
    if not is_aux_on(aux_cfg, "species"):
        return 0
    col = str(getattr(getattr(aux_cfg, "species", None), "col", "Species"))
    if col not in df.columns:
        print(f"[WARN] species col not found: {col} -> num_species=0")
        return 0
    species_list = sorted(df[col].dropna().astype(str).unique().tolist())
    return int(len(species_list))


def build_model(cfg: Any, df_all: pd.DataFrame) -> nn.Module:
    """cfgから ConvNeXtSplitCropAuxRegressor を構築する。

    Args:
        cfg: expのconfig.yaml（OmegaConf）
        df_all: pivot df全体（num_species算出に使う）

    Returns:
        model（重みはまだロードしない）
    """
    mcfg = cfg.model
    aux_cfg = getattr(cfg, "aux", None)

    num_species = compute_num_species(df_all, aux_cfg)

    model = ConvNeXtSplitCropAuxRegressor(
        backbone=str(getattr(mcfg, "backbone", "convnext_small")),
        pretrained=False,  # OOF評価はckptロード前提なのでFalse
        num_targets=len(cfg.target_cols),
        in_chans=int(getattr(mcfg, "in_chans", 3)),
        drop_rate=float(getattr(mcfg, "drop_rate", 0.0)),
        drop_path_rate=float(getattr(mcfg, "drop_path_rate", 0.0)),
        head_dropout=float(getattr(mcfg, "head_dropout", 0.0)),
        fuse=str(getattr(mcfg, "fuse", "concat")),
        aux_cfg=aux_cfg,
        num_species=int(num_species),
        aux_hidden_dim=int(getattr(mcfg, "aux_hidden_dim", 256)),
        aux_dropout=float(getattr(mcfg, "aux_dropout", 0.1)),
    )
    return model


def load_ckpt(model: nn.Module, ckpt_path: Path, device: torch.device) -> nn.Module:
    """ckptをロードして eval モードにする。"""
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

    if isinstance(ckpt, dict):
        state = ckpt.get("model_state_dict") or ckpt.get("state_dict") or ckpt.get("model") or ckpt
    else:
        state = ckpt

    # DDPの "module." を除去
    if isinstance(state, dict) and any(k.startswith("module.") for k in state.keys()):
        state = {k.replace("module.", "", 1): v for k, v in state.items()}

    model.load_state_dict(state, strict=True)
    model.to(device)
    model.eval()
    return model

In [8]:
# =========================================================
# 6. 予測（TTA + crop平均）
# =========================================================
def make_tta_views(x: torch.Tensor, tta: str) -> List[torch.Tensor]:
    """flip系TTAのビューを作る（5D/6D対応）。

    Args:
        x: (B,2,C,H,W) または (B*N,2,C,H,W) など
        tta: "none" | "hflip" | "vflip" | "hflip_vflip"

    Returns:
        xと同shapeのテンソルリスト。
    """
    tta = (tta or "none").lower()
    if tta in ("none", "off", "false", "0"):
        return [x]

    # W/H は常に末尾2次元
    dim_w = -1
    dim_h = -2

    if tta == "hflip":
        return [x, torch.flip(x, dims=[dim_w]).contiguous()]
    if tta == "vflip":
        return [x, torch.flip(x, dims=[dim_h]).contiguous()]
    if tta in ("hflip_vflip", "hvflip", "hflip+vflip"):
        x_h = torch.flip(x, dims=[dim_w]).contiguous()
        x_v = torch.flip(x, dims=[dim_h]).contiguous()
        x_hv = torch.flip(x_v, dims=[dim_w]).contiguous()
        return [x, x_h, x_v, x_hv]

    raise ValueError(f"Unknown tta={tta}")


def extract_pred_log1p(model_out: Any) -> torch.Tensor:
    """モデル出力から pred_log1p を取り出す。"""
    if isinstance(model_out, torch.Tensor):
        return model_out
    if isinstance(model_out, dict):
        return model_out["pred_log1p"]
    raise TypeError(f"Unexpected model output type: {type(model_out)}")


def log1p_to_raw(x_log: torch.Tensor, clip_min: float, clip_max: float) -> torch.Tensor:
    """log1p -> raw（expm1前にクリップ、非負クリップも行う）"""
    x_log = torch.clamp(x_log, min=float(clip_min), max=float(clip_max))
    x_raw = torch.expm1(x_log)
    x_raw = torch.clamp(x_raw, min=0.0)
    return x_raw


@torch.no_grad()
def predict_loader_raw(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    *,
    tta: str,
    avg_space: str,
    use_amp: bool,
    output_is_log1p: bool,
    log_clip_min: float,
    log_clip_max: float,
) -> Tuple[np.ndarray, np.ndarray, List[str]]:
    """loaderを回して (pred_raw, target_raw, ids) を返す。

    Args:
        model: 推論モデル
        loader: DataLoader（batch["image"] は split-crop）
        device: CUDA/CPU
        tta: TTAモード
        avg_space: "raw" or "log"
        use_amp: AMP使用
        output_is_log1p: モデル出力がlog1pか（通常True）
        log_clip_min/max: expm1前の安全クリップ

    Returns:
        preds_raw: (N,K) raw
        targs_raw: (N,K) raw
        ids_all:   len=N の id list
    """
    preds_list: List[np.ndarray] = []
    targs_list: List[np.ndarray] = []
    ids_all: List[str] = []

    amp_on = bool(use_amp and device.type == "cuda")
    pbar = tqdm(loader, total=len(loader), leave=False)

    for batch in pbar:
        x = batch["image"]  # (B,2,C,H,W) or (B,N,2,C,H,W)
        y = batch["target"] # (B,K) log1p or raw
        ids = batch["id"]

        ids_all.extend([str(v) for v in ids])

        # -----------------------
        # crop次元をフラット化
        # -----------------------
        if x.ndim == 5:
            b = x.shape[0]
            n_crops = 1
            x_in = x  # (B,2,C,H,W)
        elif x.ndim == 6:
            b, n_crops = x.shape[0], x.shape[1]
            x_in = x.reshape(b * n_crops, *x.shape[2:])  # (B*N,2,C,H,W)
        else:
            raise ValueError(f"Unexpected x.ndim={x.ndim}")

        x_in = x_in.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        # target を raw に揃える
        if output_is_log1p:
            y_raw = log1p_to_raw(y, log_clip_min, log_clip_max)
        else:
            y_raw = torch.clamp(y, min=0.0)

        # -----------------------
        # TTA
        # -----------------------
        views = make_tta_views(x_in, tta=tta)

        with torch.autocast(device_type="cuda", enabled=amp_on):
            if output_is_log1p and avg_space == "log":
                # log空間で平均 → 最後にexpm1
                pred_logs = []
                for xv in views:
                    out = model(xv)
                    pred_logs.append(extract_pred_log1p(out).float())
                pred_log_mean = torch.stack(pred_logs, dim=0).mean(dim=0)
                pred_raw = log1p_to_raw(pred_log_mean, log_clip_min, log_clip_max)
            else:
                # raw空間で平均（推論と同じ）
                pred_raws = []
                for xv in views:
                    out = model(xv)
                    pred_log = extract_pred_log1p(out).float()
                    if output_is_log1p:
                        pred_raws.append(log1p_to_raw(pred_log, log_clip_min, log_clip_max))
                    else:
                        pred_raws.append(torch.clamp(pred_log, min=0.0))
                pred_raw = torch.stack(pred_raws, dim=0).mean(dim=0)

        # crop平均
        if n_crops > 1:
            pred_raw = pred_raw.reshape(b, n_crops, -1).mean(dim=1)  # (B,K)

        preds_list.append(pred_raw.detach().cpu().numpy())
        targs_list.append(y_raw.detach().cpu().numpy())

    preds_raw = np.concatenate(preds_list, axis=0)
    targs_raw = np.concatenate(targs_list, axis=0)
    return preds_raw, targs_raw, ids_all

In [9]:
# =========================================================
# 7. OOF評価（foldごと＋全fold結合）
# =========================================================
def make_valid_loader(val_df: pd.DataFrame, cfg: Any) -> DataLoader:
    """valid用DataLoaderを作る（推論に寄せたdeterministic crop）。"""
    ds = OofSplitCropDataset(
        df=val_df,
        image_root=Path(str(cfg.input_dir)),
        target_cols=list(cfg.target_cols),
        transform=VALID_TFM,
        use_log1p_target=bool(cfg.use_log1p_target),
        crop_size=int(cfg.crop_size),
        crop_mode=CROP_MODE,
        num_crops=NUM_CROPS,
        base_seed=int(cfg.seed),
    )
    return DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=bool(PIN_MEMORY),
        drop_last=False,
    )


def eval_oof_tta(
    df: pd.DataFrame,
    cfg: Any,
    exp_dir: Path,
    folds: List[int],
    tta_modes: List[str],
) -> Dict[str, Dict[str, float]]:
    """TTAごとにOOF評価を行う（fold結合で1発計算）。

    Returns:
        results[tta] = {
            "weighted_r2_oof": ...,
            "r2_mean": ...,
            "r2_<target>": ...,
        }
    """
    results: Dict[str, Dict[str, float]] = {}
    output_is_log1p = bool(cfg.use_log1p_target)  # 通常True

    for tta in tta_modes:
        print("\n" + "=" * 60)
        print(f"TTA = {tta} (AVG_SPACE={AVG_SPACE}, CROP_MODE={CROP_MODE}, NUM_CROPS={NUM_CROPS})")
        print("=" * 60)

        preds_all = []
        targs_all = []

        for fold in folds:
            ckpt_path = exp_dir / "model" / f"best_fold{fold}.pth"
            if not ckpt_path.exists():
                raise FileNotFoundError(f"ckpt not found: {ckpt_path}")

            model = build_model(cfg, df_all=df)
            model = load_ckpt(model, ckpt_path, device=DEVICE)

            val_df = df[df[fold_col] == fold].reset_index(drop=True)
            loader = make_valid_loader(val_df, cfg)

            preds_raw, targs_raw, _ = predict_loader_raw(
                model=model,
                loader=loader,
                device=DEVICE,
                tta=tta,
                avg_space=AVG_SPACE,
                use_amp=USE_AMP,
                output_is_log1p=output_is_log1p,
                log_clip_min=LOG_CLIP_MIN,
                log_clip_max=LOG_CLIP_MAX,
            )

            r2_fold = global_weighted_r2_score(targs_raw, preds_raw, weights)
            print(f"  fold{fold}: weighted_r2={r2_fold:.6f} (n={len(val_df)})")

            preds_all.append(preds_raw)
            targs_all.append(targs_raw)

            del model
            gc.collect()
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

        preds_oof = np.concatenate(preds_all, axis=0)
        targs_oof = np.concatenate(targs_all, axis=0)

        weighted_r2 = global_weighted_r2_score(targs_oof, preds_oof, weights)
        r2_each = r2_per_target(targs_oof, preds_oof)

        out = {
            "weighted_r2_oof": float(weighted_r2),
            "r2_mean": float(np.mean(r2_each)),
        }
        for name, r2v in zip(target_cols, r2_each):
            out[f"r2_{name}"] = float(r2v)

        results[tta] = out

        print("  ---- OOF (all folds combined) ----")
        print(f"  weighted_r2_oof: {weighted_r2:.6f}")
        for name, r2v in zip(target_cols, r2_each):
            print(f"  r2_{name}: {r2v:.6f}")

    return results

In [10]:
# =========================================================
# 8. Run
# =========================================================
res = eval_oof_tta(
    df=df,
    cfg=cfg,
    exp_dir=EXP_DIR,
    folds=folds,
    tta_modes=TTA_MODES,
)

print("\n===== Summary =====")
for tta, d in res.items():
    print(tta, "=>", d["weighted_r2_oof"])
res


TTA = none (AVG_SPACE=raw, CROP_MODE=center, NUM_CROPS=1)


  fold0: weighted_r2=0.689380 (n=121)


  fold1: weighted_r2=0.666383 (n=115)


  fold2: weighted_r2=0.641189 (n=121)
  ---- OOF (all folds combined) ----
  weighted_r2_oof: 0.667930
  r2_Dry_Green_g: 0.551352
  r2_Dry_Clover_g: 0.480972
  r2_Dry_Dead_g: 0.227089
  r2_GDM_g: 0.554153
  r2_Dry_Total_g: 0.583823

TTA = hflip (AVG_SPACE=raw, CROP_MODE=center, NUM_CROPS=1)


  fold0: weighted_r2=0.691608 (n=121)


...

  fold2: weighted_r2=0.638624 (n=121)
  ---- OOF (all folds combined) ----
  weighted_r2_oof: 0.667881
  r2_Dry_Green_g: 0.551118
  r2_Dry_Clover_g: 0.481742
  r2_Dry_Dead_g: 0.217873
  r2_GDM_g: 0.554415
  r2_Dry_Total_g: 0.584010

TTA = hflip_vflip (AVG_SPACE=raw, CROP_MODE=center, NUM_CROPS=1)


  fold0: weighted_r2=0.690247 (n=121)


  fold1: weighted_r2=0.668101 (n=115)


  fold2: weighted_r2=0.639011 (n=121)
  ---- OOF (all folds combined) ----
  weighted_r2_oof: 0.668336
  r2_Dry_Green_g: 0.552067
  r2_Dry_Clover_g: 0.468342
  r2_Dry_Dead_g: 0.228525
  r2_GDM_g: 0.553757
  r2_Dry_Total_g: 0.585081

===== Summary =====
none => 0.6679301724995906
hflip => 0.6678807379205594
hflip_vflip => 0.6683360533776872


{'none': {'weighted_r2_oof': 0.6679301724995906,
  'r2_mean': 0.4794778868377314,
  'r2_Dry_Green_g': 0.5513515441362584,
  'r2_Dry_Clover_g': 0.4809722367291097,
  'r2_Dry_Dead_g': 0.22708918183949212,
  'r2_GDM_g': 0.5541531883033106,
  'r2_Dry_Total_g': 0.5838232831804862},
 'hflip': {'weighted_r2_oof': 0.6678807379205594,
  'r2_mean': 0.4778316001413188,
  'r2_Dry_Green_g': 0.5511176438198382,
  'r2_Dry_Clover_g': 0.48174206086613436,
  'r2_Dry_Dead_g': 0.217872970607762,
  'r2_GDM_g': 0.5544149253594598,
  'r2_Dry_Total_g': 0.5840104000533998},
 'hflip_vflip': {'weighted_r2_oof': 0.6683360533776872,
  'r2_mean': 0.4775546544625141,
  'r2_Dry_Green_g': 0.5520672429515134,
  'r2_Dry_Clover_g': 0.4683423103581318,
  'r2_Dry_Dead_g': 0.22852509369930607,
  'r2_GDM_g': 0.5537572630908478,
  'r2_Dry_Total_g': 0.5850813622127715}}